# PAC Report – Credit Card Monthly Spend Prediction

## 1. Project Overview

**Goal.**  
Predict each customer's **monthly credit card spend** (`monthly_spend`) using demographic, credit, and behavioral features such as age, annual income, credit score, credit limit, transaction behavior, online shopping frequency, travel, and household composition.

**This notebook.**  
This notebook documents:

- How I explored and prepared the data.
- The different modeling approaches and submissions (including what did **not** work).
- The reasoning behind my final model: a **StackingRegressor** combining Gradient Boosting and Random Forest with a Ridge meta-learner.
- Robustness checks (including a GNTest stability test with Gaussian noise).
- The **exact code** used to generate my final submission file.

Other important submission codes (OLS variants, Gradient Boosting variants, Random Forest, hybrid ensembles) are included in a separate notebook:  
`PAC_Report_OtherSubmissions.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

# General settings
RANDOM_STATE = 2101
TRAIN_PATH = "/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/analysis_data.csv"
TEST_PATH  = "/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/scoring_data.csv"   
ID_COL     = "customer_id"
TARGET_COL = "monthly_spend"

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

## 2. Data Understanding & Initial Exploration

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
train.head()The training dataset includes:

- A unique identifier: `customer_id`.
- **Predictors (21 features)** including:
  - Demographics: `age`, `gender`, `marital_status`, `education_level`, `region`, `num_children`.
  - Economic/credit: `employment_status`, `owns_home`, `has_auto_loan`, `annual_income`, `credit_score`, `credit_limit`, `tenure`, `card_type`, `num_credit_cards`.
  - Behavioral: `num_transactions`, `avg_transaction_value`, `online_shopping_freq`, `reward_points_balance`, `travel_frequency`, `utility_payment_count`.
- **Target**: `monthly_spend` (USD).

The scoring dataset has the same predictors and `customer_id`, but **no target**.

### 2.1 Target distribution (`monthly_spend`)

In [ ]:
sns.histplot(train[TARGET_COL], bins=30, kde=True)
plt.title("Distribution of Monthly Credit Card Spend")
plt.xlabel("monthly_spend")
plt.ylabel("Count")
plt.show()

train[TARGET_COL].describe()

The distribution of `monthly_spend` is **right-skewed** with a long tail of high-spending customers.  
This suggested that:

- Linear models (like OLS) would struggle with the tail.
- Tree-based models, and possibly log-transformations of the target, could better capture the non-linearity.

### 2.2 Correlations with numeric features

In [ ]:
numeric_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in [TARGET_COL, ID_COL]]

corr = train[numeric_cols + [TARGET_COL]].corr()[TARGET_COL].sort_values(ascending=False)
corr